<a href="https://colab.research.google.com/github/base76-research-lab/.github/blob/main/control_block_gpt2medium_readonly_colab_2026_03_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Block — GPT-2 Medium, Read-Only Observer
**Run:** `control_block_gpt2_medium_readonly_2026-03-12`  
**Model:** `gpt2-medium`  
**Mode:** read-only observer (no write-back, no intervention)  
**Layers:** 10, 12, 14, 16  
**SAE:** bootstrap l12 seed (same as 2026-03-11 runs)  
**Goal:** Replikera 2026-03-11 observer-pass med ny seed/datum för cross-day stabilitet

> Kör i Colab med GPU-runtime (T4 räcker). Runtime → Change runtime type → T4 GPU.

## 0. GPU-check

In [1]:
import shutil
import subprocess

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    result = subprocess.run([nvidia_smi], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print('nvidia-smi saknas i denna runtime. Fortsätt till installationscellen och verifiera CUDA där istället.')

Thu Mar 12 08:22:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Klona repo

In [2]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/base76-research-lab/Mechanistic-Interpretability'

def find_existing_repo_root(start: Path) -> Path | None:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'transformer_oscilloscope').exists() and (candidate / 'scripts').exists():
            return candidate
    return None

existing_root = find_existing_repo_root(Path.cwd())
if existing_root is not None:
    REPO_DIR = str(existing_root)
    print('Använder befintlig lokal repo:', REPO_DIR)
else:
    base_dir = Path('/content') if Path('/content').exists() and os.access('/content', os.W_OK) else Path.cwd()
    REPO_DIR = str(base_dir / 'mi')
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    print('Klonad repo till:', REPO_DIR)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

Cloning into '/content/mi'...
remote: Enumerating objects: 3787, done.
remote: Counting objects: 100% (3787/3787), done.
remote: Compressing objects: 100% (1676/1676), done.
remote: Total 3787 (delta 2216), reused 3648 (delta 2085), pack-reused 0 (from 0)
Receiving objects: 100% (3787/3787), 13.68 MiB | 11.27 MiB/s, done.
Resolving deltas: 100% (2216/2216), done.
Klonad repo till: /content/mi
cwd: /content/mi


## 2. Installera beroenden

In [3]:
import importlib.util
import sys

# Installera bara det som behövs för denna körning och undvik att skriva över Colabs kärnpaket.
if importlib.util.find_spec('torch') is None:
    !{sys.executable} -m pip install -q torch

!{sys.executable} -m pip install -q 'transformers>=4.41,<5'

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('Ingen CUDA synlig. I Colab: Runtime -> Change runtime type -> GPU, sedan reconnect och kör om från början.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.6 MB/s eta 0:00:00
torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA A100-SXM4-40GB


In [4]:
from pathlib import Path

DRIVE_SAE_STATE = None

if Path('/content').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)

        hits = sorted(Path('/content/drive').glob('**/sae_weights.pt'))
        print(f'found sae_weights.pt in Drive: {len(hits)}')
        for p in hits[:20]:
            print(p)

        # Prioritera tydliga gpt2-medium paths om de finns.
        medium_hits = [p for p in hits if 'gpt2medium' in str(p).lower() or 'gpt2_medium' in str(p).lower()]
        chosen = medium_hits[-1] if medium_hits else (hits[-1] if hits else None)
        DRIVE_SAE_STATE = str(chosen) if chosen else None
        print('auto-selected DRIVE_SAE_STATE:', DRIVE_SAE_STATE)
    except Exception as e:
        print('Drive-mount hoppades over / misslyckades:', e)
else:
    print('Inte i Colab-runtime. Drive-mount hoppas over.')

Mounted at /content/drive
found sae_weights.pt in Drive: 0
auto-selected DRIVE_SAE_STATE: None


## 2.5 Mount Drive + hitta nya SAE-weights (Colab)

## 3. Konfigurera körning

In [5]:
import datetime
from pathlib import Path

# -- Parametrar (andra vid behov) --
MODEL         = 'gpt2-medium'
LAYERS        = [10, 12, 14, 16]
UNITS         = [472, 468, 57, 156, 346]
BASIS_MODE    = 'pc2'
SAE_TOPK      = 8
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

ROOT      = Path(REPO_DIR)
PANEL     = ROOT / 'data' / 'prompts_observability_panel_expanded_2026-03-10.jsonl'
OUT_ROOT  = ROOT / 'experiments' / 'exp_005_trajectory_block'

# Prioritet: explicit override -> auto-vald Drive-vikt -> auto-detektion i repo.
SAE_STATE_OVERRIDE = None
AUTO_DRIVE_SAE = globals().get('DRIVE_SAE_STATE', None)

if SAE_STATE_OVERRIDE:
    SAE_STATE = Path(SAE_STATE_OVERRIDE)
elif AUTO_DRIVE_SAE:
    SAE_STATE = Path(AUTO_DRIVE_SAE)
else:
    sae_candidates = sorted(
        p for p in ROOT.glob('experiments/**/sae_weights.pt')
        if 'gpt2medium' in str(p).lower() or 'gpt2_medium' in str(p).lower()
    )
    if not sae_candidates:
        sae_candidates = sorted(ROOT.glob('experiments/**/sae_weights.pt'))
    SAE_STATE = sae_candidates[-1] if sae_candidates else ROOT / 'experiments' / 'MISSING' / 'sae_weights.pt'

USE_SAE = SAE_STATE.exists()

DATE_TAG  = datetime.date.today().isoformat().replace('-', '')
RUN_NAME  = f'control_block_gpt2_medium_readonly_{DATE_TAG}'

print('model  :', MODEL)
print('layers :', LAYERS)
print('device :', DEVICE)
print('root   :', ROOT)
print('run_name:', RUN_NAME)
print('drive weight:', AUTO_DRIVE_SAE)
print('panel exists :', PANEL.exists(), PANEL)
print('sae exists   :', SAE_STATE.exists(), SAE_STATE)
print('use_sae      :', USE_SAE)
if DEVICE != 'cuda':
    print('Varning: runtime kor pa CPU. I Colab: Runtime -> Change runtime type -> GPU och kor om fran borjan.')
if not PANEL.exists():
    raise FileNotFoundError(f'Panel saknas: {PANEL}')
if not USE_SAE:
    print('Info: Ingen sae_weights.pt hittades. Kor read-only observer utan SAE-baserade metrics.')

model  : gpt2-medium
layers : [10, 12, 14, 16]
device : cuda
root   : /content/mi
run_name: control_block_gpt2_medium_readonly_20260312
drive weight: None
panel exists : True /content/mi/data/prompts_observability_panel_expanded_2026-03-10.jsonl
sae exists   : False /content/mi/experiments/MISSING/sae_weights.pt
use_sae      : False
Info: Ingen sae_weights.pt hittades. Kor read-only observer utan SAE-baserade metrics.


## 3.5 Patcha trace.py device-bugg (Colab-klon)

In [6]:
from pathlib import Path

trace_py = Path(REPO_DIR) / 'transformer_oscilloscope' / 'trace.py'
src = trace_py.read_text()

old = 'coords = torch.tensor([c["coords"] for c in candidates], dtype=torch.float32)'
new = (
    'coords_device = field_coords.device if field_coords is not None else None\n'
    '    coords = torch.tensor([c["coords"] for c in candidates], dtype=torch.float32, device=coords_device)'
)

if old in src:
    trace_py.write_text(src.replace(old, new))
    print('Patch applied:', trace_py)
else:
    print('Patch already present or code changed:', trace_py)

Patch applied: /content/mi/transformer_oscilloscope/trace.py


## 4. Kör read-only observer

In [7]:
import time
import torch

# Re-evaluate device at run time to avoid stale notebook state.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('runtime DEVICE:', DEVICE)
if DEVICE != 'cuda':
    raise RuntimeError('Den här körningen är avsedd för Colab GPU, men runtime är fortfarande CPU. Byt till GPU och kör om från början.')

cmd = [
    'python3', '-m', 'transformer_oscilloscope.cli', 'trace',
    '--prompt-jsonl',  str(PANEL),
    '--model',         MODEL,
    '--layers',        *[str(l) for l in LAYERS],
    '--device',        DEVICE,
    '--out-dir',       str(OUT_ROOT / RUN_NAME),
    '--run-name',      f'{RUN_NAME}_readonly',
    '--store-projections',
]

if USE_SAE:
    cmd += [
        '--sae-state', str(SAE_STATE),
        '--sae-topk', str(SAE_TOPK),
        '--basis-mode', BASIS_MODE,
        '--units', *[str(u) for u in UNITS],
    ]
else:
    print('Kor utan SAE-flaggor (--sae-state saknas).')

print('Startar:', ' '.join(cmd))
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))
elapsed = round(time.time() - t0, 1)

print(result.stdout[-3000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError(f'Run failed (returncode={result.returncode})')

print(f'\nKlar. Wall-clock: {elapsed}s')

runtime DEVICE: cuda
Kor utan SAE-flaggor (--sae-state saknas).
Startar: python3 -m transformer_oscilloscope.cli trace --prompt-jsonl /content/mi/data/prompts_observability_panel_expanded_2026-03-10.jsonl --model gpt2-medium --layers 10 12 14 16 --device cuda --out-dir /content/mi/experiments/exp_005_trajectory_block/control_block_gpt2_medium_readonly_20260312 --run-name control_block_gpt2_medium_readonly_20260312_readonly --store-projections
Saved trace: /content/mi/experiments/exp_005_trajectory_block/control_block_gpt2_medium_readonly_20260312/control_block_gpt2_medium_readonly_20260312_readonly/trace.jsonl


Klar. Wall-clock: 40.4s


## 5. Verifiera output

In [8]:
import json

trace_dir = OUT_ROOT / RUN_NAME / f'{RUN_NAME}_readonly'
trace_file = trace_dir / 'trace.jsonl'
meta_file  = trace_dir / 'metadata.json'

print('trace.jsonl exists:', trace_file.exists())
print('metadata.json exists:', meta_file.exists())

if trace_file.exists():
    lines = trace_file.read_text().strip().splitlines()
    print(f'Antal traces: {len(lines)}')
    print('Exempel (rad 1):', json.loads(lines[0]).keys() if lines else 'tom')

if meta_file.exists():
    meta = json.loads(meta_file.read_text())
    print('Metadata:', json.dumps(meta, indent=2))

trace.jsonl exists: True
metadata.json exists: True
Antal traces: 1716
Exempel (rad 1): dict_keys(['prompt_id', 'regime', 'layer', 'token_index', 'token', 'entropy', 'lens_entropy', 'gap_top2', 'entropy_delta_vs_prev_token', 'topk', 'lens_topk', 'hidden_sha', 'mlp_sha', 'attn_entropy', 'pca_x', 'pca_y'])
Metadata: {
  "run_name": "control_block_gpt2_medium_readonly_20260312_readonly",
  "model": "gpt2-medium",
  "layers": [
    10,
    12,
    14,
    16
  ],
  "device": "cuda",
  "panel": "/content/mi/data/prompts_observability_panel_expanded_2026-03-10.jsonl",
  "record_count": 1716,
  "store_projections": true,
  "sae_state": null,
  "sae_topk": 8,
  "units": null,
  "basis_mode": null,
  "trace_file": "/content/mi/experiments/exp_005_trajectory_block/control_block_gpt2_medium_readonly_20260312/control_block_gpt2_medium_readonly_20260312_readonly/trace.jsonl",
  "observer_class": "read_only"
}


## 6. Logg (kopiera till SESSION.md)

```
run_id: control_block_gpt2_medium_readonly_20260312
model:  gpt2-medium
layers: [10, 12, 14, 16]
device: cuda (Colab T4)
result: success/fail
traces: <antal>
wall_clock: <sekunder>
```